# Stage 5 — Experiment Tracking with MLflow

**What this notebook covers:**

| Part | Task |
|---|---|
| 1 | Setup & Configuration — MLflow tracking URI, experiment |
| 2 | Load Data & Trained Models from Stages 3 & 4 |
| 3 | Log Parameters & Metrics for both models |
| 4 | Log Model Artefacts to MLflow |
| 5 | Register Models in the MLflow Model Registry |
| 6 | Set `@champion` / `@challenger` aliases (2026 API — replaces deprecated `stage=Production`) |
| 7 | Load Model by Alias and verify predictions |
| 8 | Compare Runs Programmatically |

**Inputs**: `models/titanic_model_v1.pkl`, `models/titanic_model_v2.pkl`, `data/processed/`  
**MLflow version**: 3.10.1  
**Output**: `mlruns/` experiment store + registered model `titanic-classifier`

> ⚠️ Run `mlflow ui` in the project root terminal before opening http://127.0.0.1:5000


---
## Part 1 — MLflow Concepts

### Why Experiment Tracking?

In classical software development, code is versioned with Git. In ML, you also need to version:
- **Parameters**: model type, hyperparameters, preprocessing choices
- **Metrics**: Accuracy, F1, AUC — on train, val, and test sets
- **Artefacts**: the trained model file, feature pipeline, plots

Without tracking, you cannot answer: *"Which combination of hyperparameters produced this model?"*

### MLflow Core Components

| Component | Purpose |
|---|---|
| **Tracking Server** | Records runs to a local `mlruns/` folder (or remote DB) |
| **Experiment** | A named group of related runs (e.g. `titanic-classifier`) |
| **Run** | One training execution — captures params, metrics, artefacts |
| **Model Registry** | Versioned store of promoted model artefacts |
| **Alias** | Named pointer to a model version (`@champion`, `@challenger`) |

### 2026 API: Aliases vs Deprecated Stages

| Old (deprecated) | New (2026) |
|---|---|
| `client.transition_model_version_stage(name, version, stage="Production")` | `client.set_registered_model_alias(name, alias="champion", version="1")` |
| `mlflow.pyfunc.load_model("models:/name/Production")` | `mlflow.pyfunc.load_model("models:/name@champion")` |

**Never use `stage=Production`** in new code — it is deprecated in MLflow 3.x.


In [1]:
# Install / verify MLflow in the active kernel
%pip install -q mlflow


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import warnings
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report
)

warnings.filterwarnings('ignore')

print(f"mlflow version : {mlflow.__version__}")

# ── Tracking URI ──────────────────────────────────────────────────────────────
# Use the local MLflow server if it's running; fall back to file store otherwise.
# To start the server: run `mlflow ui` in the project root terminal.
TRACKING_URI = "http://127.0.0.1:5000"
try:
    import urllib.request
    urllib.request.urlopen(TRACKING_URI, timeout=2)
    mlflow.set_tracking_uri(TRACKING_URI)
    print(f"Connected to MLflow server at {TRACKING_URI}")
except Exception:
    mlflow.set_tracking_uri("../mlruns")   # local file store fallback
    print("MLflow server not running — using local file store (../mlruns)")

# ── Experiment ────────────────────────────────────────────────────────────────
EXPERIMENT_NAME = "titanic-classifier"
REGISTERED_MODEL = "titanic-classifier"

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: '{EXPERIMENT_NAME}'")


2026/03/28 05:09:40 INFO mlflow.tracking.fluent: Experiment with name 'titanic-classifier' does not exist. Creating a new experiment.


mlflow version : 3.10.1
Connected to MLflow server at http://127.0.0.1:5000
Experiment: 'titanic-classifier'


---
## Part 2 — Load Data & Trained Models


In [3]:
processed_dir = '../data/processed'
models_dir    = '../models'

X_train = pd.read_parquet(os.path.join(processed_dir, 'X_train.parquet'))
X_test  = pd.read_parquet(os.path.join(processed_dir, 'X_test.parquet'))
y_train = pd.read_csv(os.path.join(processed_dir, 'y_train.csv'), index_col=0).squeeze()
y_test  = pd.read_csv(os.path.join(processed_dir, 'y_test.csv'),  index_col=0).squeeze()

# always cast bool → int (known Parquet quirk from Stage 2)
X_train = X_train.assign(alone=X_train['alone'].astype(int))
X_test  = X_test.assign(alone=X_test['alone'].astype(int))

# Load both models
model_v1 = joblib.load(os.path.join(models_dir, 'titanic_model_v1.pkl'))  # Stage 3 baseline
model_v2 = joblib.load(os.path.join(models_dir, 'titanic_model_v2.pkl'))  # Stage 4 best tuned

print(f"X_test shape  : {X_test.shape}")
print(f"model_v1 type : {type(model_v1).__name__}")
print(f"model_v2 type : {type(model_v2).__name__}")

# Precompute predictions + probabilities for both models
preds_v1  = model_v1.predict(X_test)
proba_v1  = model_v1.predict_proba(X_test)[:, 1]

preds_v2  = model_v2.predict(X_test)
proba_v2  = model_v2.predict_proba(X_test)[:, 1]

def compute_metrics(y_true, y_pred, y_proba):
    return {
        'accuracy' : round(accuracy_score(y_true, y_pred),  4),
        'f1'       : round(f1_score(y_true, y_pred),         4),
        'auc_roc'  : round(roc_auc_score(y_true, y_proba),   4),
    }

metrics_v1 = compute_metrics(y_test, preds_v1, proba_v1)
metrics_v2 = compute_metrics(y_test, preds_v2, proba_v2)

print(f"\nv1 metrics: {metrics_v1}")
print(f"v2 metrics: {metrics_v2}")


X_test shape  : (157, 14)
model_v1 type : LogisticRegression
model_v2 type : XGBClassifier

v1 metrics: {'accuracy': 0.828, 'f1': 0.797, 'auc_roc': 0.8874}
v2 metrics: {'accuracy': 0.8408, 'f1': 0.7934, 'auc_roc': 0.8942}


---
## Part 3 — Log Parameters, Metrics & Artefacts

Each model gets its own MLflow **run**. Within each run we log:
- `mlflow.log_param()` — hyperparameters extracted from `model.get_params()`
- `mlflow.log_metric()` — Accuracy, F1, AUC-ROC on the held-out test set
- `mlflow.sklearn.log_model()` — the sklearn model serialised as an MLflow artefact, and **registered** in the Model Registry in the same call
- `mlflow.log_artifact()` — the feature pipeline pkl

> Setting `registered_model_name` inside `log_model()` is the recommended single-step approach — no need for a separate `mlflow.register_model()` call.


In [4]:
PIPELINE_PATH = os.path.join(models_dir, 'feature_pipeline.pkl')

def log_model_run(model, metrics, run_name, tags=None):
    """Log one model as a full MLflow run and register it."""
    with mlflow.start_run(run_name=run_name, tags=tags or {}) as run:
        # ── Parameters ────────────────────────────────────────────────────────
        mlflow.log_param('model_type', type(model).__name__)
        for k, v in model.get_params().items():
            mlflow.log_param(k, v)

        # ── Metrics ────────────────────────────────────────────────────────────
        mlflow.log_metric('accuracy', metrics['accuracy'])
        mlflow.log_metric('f1',       metrics['f1'])
        mlflow.log_metric('auc_roc',  metrics['auc_roc'])

        # ── Model artefact + register in one step ─────────────────────────────
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path='model',
            registered_model_name=REGISTERED_MODEL,
        )

        # ── Feature pipeline as extra artefact ────────────────────────────────
        mlflow.log_artifact(PIPELINE_PATH, artifact_path='pipeline')

        run_id = run.info.run_id

    print(f"Logged run '{run_name}'  |  run_id: {run_id}")
    print(f"  Accuracy={metrics['accuracy']}  F1={metrics['f1']}  AUC={metrics['auc_roc']}")
    return run_id


# Log Stage 3 baseline
run_id_v1 = log_model_run(
    model_v1, metrics_v1,
    run_name='v1-logistic-regression-baseline',
    tags={'stage': 'stage3', 'tuning': 'none'},
)


2026/03/28 05:09:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 05:09:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'titanic-classifier'.
2026/03/28 05:09:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: titanic-classifier, version 1


🏃 View run v1-logistic-regression-baseline at: http://127.0.0.1:5000/#/experiments/1/runs/82e2b6e3ce0a46509926988c7efa4727
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Logged run 'v1-logistic-regression-baseline'  |  run_id: 82e2b6e3ce0a46509926988c7efa4727
  Accuracy=0.828  F1=0.797  AUC=0.8874


Created version '1' of model 'titanic-classifier'.


In [5]:
# Log Stage 4 best tuned model
run_id_v2 = log_model_run(
    model_v2, metrics_v2,
    run_name='v2-tuned-best-model',
    tags={'stage': 'stage4', 'tuning': 'optuna'},
)


2026/03/28 05:09:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 05:09:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'titanic-classifier' already exists. Creating a new version of this model...
2026/03/28 05:09:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: titanic-classifier, version 2


🏃 View run v2-tuned-best-model at: http://127.0.0.1:5000/#/experiments/1/runs/cb8e4163acd94a00a5cc2757de84888b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Logged run 'v2-tuned-best-model'  |  run_id: cb8e4163acd94a00a5cc2757de84888b
  Accuracy=0.8408  F1=0.7934  AUC=0.8942


Created version '2' of model 'titanic-classifier'.


---
## Part 4 — Inspect Registered Model Versions

Both runs registered their model under `titanic-classifier`. Let's inspect the versions in the registry.


In [6]:
client = MlflowClient()

versions = client.search_model_versions(f"name='{REGISTERED_MODEL}'")
print(f"Registered model '{REGISTERED_MODEL}' — {len(versions)} version(s):\n")
for v in versions:
    print(f"  Version {v.version}  |  run_id: {v.run_id[:8]}...  |  source: {v.source}")


Registered model 'titanic-classifier' — 2 version(s):

  Version 2  |  run_id: cb8e4163...  |  source: models:/m-0b872f3876a4484296c21c15e74fdf02
  Version 1  |  run_id: 82e2b6e3...  |  source: models:/m-2b43ea7fdd2f41828e6947d289c6a64a


---
## Part 5 — Set Model Aliases: `@champion` and `@challenger`

**Why aliases?**

In MLflow 3.x, aliases are the official way to mark a model version for a role:
- `@champion` = the model currently serving production traffic
- `@challenger` = a new candidate being evaluated for promotion

The old `stage=Production` API is deprecated and will be removed — never use it in new projects.

**Promotion workflow:**
1. Train and evaluate `@challenger` candidate
2. If metrics meet the bar → run `set_registered_model_alias(..., "champion", new_version)`
3. `@champion` pointer moves automatically — no redeployment needed if your serving layer reads by alias


In [7]:
# Determine version numbers from the registry
# Version 1 = v1 run (Stage 3 baseline), Version 2 = v2 run (Stage 4 tuned)
# If the registry is freshly populated, versions are 1 and 2 in order of registration.
all_versions = client.search_model_versions(f"name='{REGISTERED_MODEL}'")
all_versions_sorted = sorted(all_versions, key=lambda v: int(v.version))

version_champion  = all_versions_sorted[0].version   # v1 = current champion
version_challenger = all_versions_sorted[-1].version  # v2 = challenger (newest)

# ── Set @champion on v1 (Stage 3 baseline — currently in production) ─────────
client.set_registered_model_alias(
    name=REGISTERED_MODEL,
    alias='champion',
    version=version_champion,
)

# ── Set @challenger on v2 (Stage 4 tuned — candidate for promotion) ──────────
client.set_registered_model_alias(
    name=REGISTERED_MODEL,
    alias='challenger',
    version=version_challenger,
)

print(f"@champion  → version {version_champion}  (Stage 3 Logistic Regression, F1={metrics_v1['f1']})")
print(f"@challenger → version {version_challenger}  (Stage 4 tuned model,          F1={metrics_v2['f1']})")
print("\nAliases set successfully ✓")
print("View in MLflow UI → Models tab → titanic-classifier")


@champion  → version 1  (Stage 3 Logistic Regression, F1=0.797)
@challenger → version 2  (Stage 4 tuned model,          F1=0.7934)

Aliases set successfully ✓
View in MLflow UI → Models tab → titanic-classifier


---
## Part 6 — Load Model by Alias

This is the **production pattern**: your serving layer never hardcodes a version number — it always loads `@champion`. When you promote a new model, the alias pointer moves and the serving layer automatically picks it up on next load.


In [8]:
# Load @champion by alias — this is what a production API would do
champion_uri    = f"models:/{REGISTERED_MODEL}@champion"
challenger_uri  = f"models:/{REGISTERED_MODEL}@challenger"

champion_model   = mlflow.pyfunc.load_model(champion_uri)
challenger_model = mlflow.pyfunc.load_model(challenger_uri)

print(f"Loaded champion   : {champion_uri}")
print(f"Loaded challenger : {challenger_uri}")

# Run predictions on the first 5 test rows
sample = X_test.iloc[:5]
champ_preds  = champion_model.predict(sample)
chall_preds  = challenger_model.predict(sample)

print(f"\nSample predictions (first 5 passengers):")
print(f"  True labels  : {list(y_test.iloc[:5].values)}")
print(f"  @champion    : {list(champ_preds)}")
print(f"  @challenger  : {list(chall_preds)}")


Loaded champion   : models:/titanic-classifier@champion
Loaded challenger : models:/titanic-classifier@challenger

Sample predictions (first 5 passengers):
  True labels  : [np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1)]
  @champion    : [np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1)]
  @challenger  : [np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1)]


---
## Part 7 — Compare Runs Programmatically

`mlflow.search_runs()` returns all runs as a pandas DataFrame — perfect for building a comparison table without opening the UI.


In [9]:
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.f1 DESC"],
)

# Select the columns that matter for comparison
cols = ['tags.mlflow.runName', 'params.model_type',
        'metrics.accuracy', 'metrics.f1', 'metrics.auc_roc',
        'tags.stage', 'tags.tuning', 'run_id']
available = [c for c in cols if c in runs_df.columns]

comparison = runs_df[available].rename(columns={
    'tags.mlflow.runName': 'Run Name',
    'params.model_type':   'Model Type',
    'metrics.accuracy':    'Accuracy',
    'metrics.f1':          'F1',
    'metrics.auc_roc':     'AUC-ROC',
    'tags.stage':          'Stage',
    'tags.tuning':         'Tuning',
})
comparison['run_id'] = comparison['run_id'].str[:8] + '...'

print("All runs — sorted by F1 descending:\n")
print(comparison.to_string(index=False))

best = comparison.iloc[0]
print(f"\n✅ Best run: '{best['Run Name']}'  |  F1={best['F1']}  |  AUC={best['AUC-ROC']}")


All runs — sorted by F1 descending:

                       Run Name         Model Type  Accuracy     F1  AUC-ROC  Stage Tuning      run_id
v1-logistic-regression-baseline LogisticRegression    0.8280 0.7970   0.8874 stage3   none 82e2b6e3...
            v2-tuned-best-model      XGBClassifier    0.8408 0.7934   0.8942 stage4 optuna cb8e4163...

✅ Best run: 'v1-logistic-regression-baseline'  |  F1=0.797  |  AUC=0.8874


---
## Part 8 — MLflow UI Walkthrough

After running this notebook, open the MLflow UI to explore the results visually:

```bash
# In the project root terminal:
mlflow ui
# Then open: http://127.0.0.1:5000
```

### What to look for in the UI

| UI Section | What you'll find |
|---|---|
| **Experiments → titanic-classifier** | All logged runs as rows |
| **Runs table** | Sort/filter by F1, AUC, accuracy |
| **Run detail → Parameters** | All hyperparameters that were logged |
| **Run detail → Metrics** | Accuracy, F1, AUC with their values |
| **Run detail → Artefacts** | Logged model + feature pipeline pkl |
| **Models → titanic-classifier** | Registered versions with `@champion` and `@challenger` badges |

### Promotion workflow (when @challenger beats @champion)

```python
from mlflow import MlflowClient
client = MlflowClient()

# Check if challenger beats champion
if challenger_f1 > champion_f1 + 0.005:   # only promote if gap > 0.5%
    client.set_registered_model_alias(
        name='titanic-classifier',
        alias='champion',
        version=challenger_version,     # challenger becomes new champion
    )
    print("Challenger promoted to @champion ✓")
```

---
## Stage 5 — Summary

| What we did | Key learning |
|---|---|
| Logged params with `log_param()` | Every run is fully reproducible — you know exactly what config produced each result |
| Logged metrics with `log_metric()` | Compare runs without re-running code — sorting by F1 tells you the winner |
| Logged model artefact | Model stored alongside its metadata — no more "which pickle is this?" |
| Registered model versions | Central registry = single source of truth for all promoted models |
| Set `@champion` / `@challenger` | Aliases decouple deployment from version numbers — swap models without config changes |
| Loaded model by alias | Serving code is version-agnostic: always loads `@champion`, auto-updates on promotion |
| `search_runs()` for comparison | Programmatic comparison = you can gate CI/CD on metric thresholds |

### Stage 6 preview — Model Serving with FastAPI
Next: wrap `@champion` in a FastAPI `POST /predict` endpoint that loads the model by alias at startup.
